In [16]:
import os
import openmc
import numpy as np
from IPython.display import Image, display
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm , Normalize

In [17]:
def load_parameters(filename='parametri.txt'):
    """
    Legge i parametri dal file txt e li restituisce come un dizionario.
    """
    params = {}
    # Forniamo np al contesto di esecuzione per gestire np.sqrt()
    context = {'np': np} 
    
    with open(filename, 'r') as f:
        code = f.read()
        exec(code, context, params)
    
    # Rimuoviamo 'np' e altri built-in dal dizionario finale
    return {k: v for k, v in params.items() if not k.startswith('__') and k != 'np'}

p = load_parameters('../../parametri.txt')


# --- Parametri Geometrici ---
R_CORE    = p['R_CORE']
REF_SIDE  = p['REF_SIDE']
REF_BOT   = p['REF_BOT']
H_CORE    = p['H_CORE']

# --- Parametri Logici ---
R_PIPE    = p['R_PIPE']

# --- Parametri Pin/Lattice ---
R_FUEL    = p['R_FUEL']
CLAD_THICK= p['CLAD_THICK']
R_WATER_THICK = p['R_WATER_THICK']
R_PIN     = p['R_PIN']
PITCH     = p['PITCH']

# --- Parametri Target ---
R_TARGET  = p['R_TARGET']
H_TARGET  = p['H_TARGET']
P_TARGET  = p['P_TARGET']

# --- Sorgente e Simulazione ---
S             = p['S']

bin_ris       = p['bin_ris']
batches       = p['batches_fix']

In [18]:
#Apertura delle tallies
sp = openmc.StatePoint(f'statepoint_arr_ext_5_pitch_1.h5')

Calcolo del volume del core del reattore per la mesh_z 

In [19]:
def count_fuel_pins(num_rings, pitch, r_pipe, r_core):
    total_pins = 0
    # Ring 0 è il centro
    for i in range(num_rings):
        ring_radius = i * pitch
        # Se l'anello è fuori dalla pipe centrale e dentro il raggio del core
        if ring_radius >= r_pipe and ring_radius <= r_core:
            if i == 0:
                total_pins += 1
            else:
                total_pins += (6 * i)
    return total_pins

# 1. Calcolo del numero di barre (usando la tua funzione già definita)
n_barre = count_fuel_pins(int(np.ceil(R_CORE / PITCH)) + 1, PITCH, R_PIPE, R_CORE)

# 2. Calcolo del volume analitico reale del bin
A_pin = (3 * np.sqrt(3) / 2) * (R_PIN**2)
A_core = n_barre * A_pin
dz = H_CORE / bin_ris
V = A_core * dz

print(f"Numero di barre: {n_barre}")
print(f"Area totale del core: {A_core:.2f} cm^2")
print(f"Volume reale del reattore: {V*bin_ris:.2f} cm^3")

Numero di barre: 390
Area totale del core: 4258.18 cm^2
Volume reale del reattore: 340654.56 cm^3


In [20]:

tally_fiss = sp.get_tally(name='fission_tot')
# Prendo il valore medio dello score 'fission' (fissioni per particella sorgente)
fission_per_source_particle = tally_fiss.get_slice(scores=['fission']).mean.flatten()[0]

E_f_joule = 200.0 * 1e6 * 1.6022e-19 # Energia per fissione in Joule (~3.204e-11 J)

# Calcolo Tasso di Fissione e Potenza
tasso_fissione = fission_per_source_particle * S # [fissioni / s]
potenza_termica_W = tasso_fissione * E_f_joule    # [Watt]
potenza_termica_MW = potenza_termica_W / 1e6      # [MW]

print(f"Fissioni per particella sorgente: {fission_per_source_particle:.5e}")
print(f"Tasso di fissione reale:          {tasso_fissione:.5e} fissioni/s")
print(f"Potenza termica totale:           {potenza_termica_MW:.3f} MW")

# COP = Potenza termica del reattore / Potenza elettrica sincrotrone
potenza_elettrica_sincrotrone = 0.2 # dato di riferimento paper lorenzo per avere il valore di S impostato
efficienza = 0.5 # Efficienza di conversione da termica a elettrica (ipotetica)
COP = potenza_termica_MW / (potenza_elettrica_sincrotrone / efficienza)    
print(f"Q-factor (plasma physic):         {COP:.3f}")

Fissioni per particella sorgente: 1.37078e-01
Tasso di fissione reale:          2.76897e+16 fissioni/s
Potenza termica totale:           0.887 MW
Q-factor (plasma physic):         2.218
